In [1]:
# ============================================================
# SETUP: CUDA allocator config (MUST be set before torch import)
# ============================================================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [2]:
# ============================================================
# SETUP: Install required packages
# ============================================================
!pip install -q -U transformers accelerate bitsandbytes sentencepiece
!pip install -q -U datasets huggingface_hub requests gdown
!python -m spacy download en_core_web_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
# ============================================================
# SETUP: Core imports, seed, suppress noisy bitsandbytes logging
# ============================================================
import gc, json, random, requests, shutil, pickle, sys, logging
from datetime import datetime, timezone
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

print(torch.__version__, torch.version.cuda, torch.cuda.is_available())

2.11.0+cu130 13.0 True


In [2]:
# ============================================================
# SETUP: Hugging Face auth + Drive mount
# ============================================================
import os
from huggingface_hub import login
from google.colab import userdata, drive

HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face.")
else:
    print("WARNING: HF_TOKEN not found in Colab Secrets.")

drive.mount('/content/drive')
RESULTS_DIR = "/content/drive/MyDrive/thesis_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

Logged in to Hugging Face.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ============================================================
# UTILITY: Persistent results storage (Drive-backed)
# ============================================================
def save_results(name, obj):
    path = os.path.join(RESULTS_DIR, f"{name}.pkl")
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"Saved {name} -> {path} ({datetime.now().strftime('%H:%M:%S')})")

def load_results(name):
    path = os.path.join(RESULTS_DIR, f"{name}.pkl")
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    return None

In [4]:
# ============================================================
# SETUP: Quantization config, model registry, hardware-adaptive loader
# ============================================================
BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

MODEL_REGISTRY = {
    "gemma-3-12b":   "google/gemma-3-12b-it",
    "phi-4":         "microsoft/phi-4",
    "gpt-oss-20b":   "openai/gpt-oss-20b",
    "judge-llama31": "meta-llama/Llama-3.1-8B-Instruct",
}
SUBJECT_MODEL_KEYS = ["gemma-3-12b", "phi-4", "gpt-oss-20b"]
JUDGE_MODEL_KEY = "judge-llama31"

# gpt-oss-20b: native MXFP4 needs compute capability >=9.0 (Hopper+), unavailable
# here. Two quantization-conversion attempts (bnb-4bit via a third-party repack;
# bnb-8bit via dequantize-and-resave) both silently corrupted MoE expert weights.
# bf16+CPU-offload is the only verified-correct path, at a real speed cost.
NATIVE_QUANT_MODELS = {"gpt-oss-20b"}

_loaded_model = None
_loaded_tokenizer = None
_loaded_key = None

def _offload_budget():
    import psutil
    gpu_total_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
    gpu_budget = max(int(gpu_total_gib - 10), 8)
    cpu_total_gib = psutil.virtual_memory().total / 2**30
    cpu_budget = max(int(cpu_total_gib - 8), 4)
    return {0: f"{gpu_budget}GiB", "cpu": f"{cpu_budget}GiB"}

def load_model(model_key: str):
    global _loaded_model, _loaded_tokenizer, _loaded_key

    if _loaded_key == model_key:
        return _loaded_model, _loaded_tokenizer

    unload_model()
    checkpoint = MODEL_REGISTRY[model_key]
    print(f"Loading {model_key} ({checkpoint})...")

    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if model_key in NATIVE_QUANT_MODELS:
        max_memory = _offload_budget()
        print(f"  (bf16 with CPU offload, max_memory={max_memory})")
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        model = AutoModelForCausalLM.from_pretrained(
            checkpoint, device_map="auto", max_memory=max_memory, dtype=torch.bfloat16,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            checkpoint, quantization_config=BNB_CONFIG, device_map={"": 0}, dtype=torch.bfloat16,
        )
    model.eval()
    _loaded_model, _loaded_tokenizer, _loaded_key = model, tokenizer, model_key
    return model, tokenizer

def unload_model():
    global _loaded_model, _loaded_tokenizer, _loaded_key
    if _loaded_model is not None:
        del _loaded_model, _loaded_tokenizer
        _loaded_model, _loaded_tokenizer, _loaded_key = None, None, None
        gc.collect(); torch.cuda.empty_cache()

def gpu_status():
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB / "
              f"Reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")
    else:
        print("WARNING: No GPU detected.")

gpu_status()

GPU: NVIDIA A100-SXM4-40GB
Allocated: 0.00 GB / Reserved: 0.00 GB


In [5]:
# ============================================================
# SETUP: Chat template formatting
# ============================================================
def _build_chat_prompt(tokenizer, content, model_key=None):
    messages = [{"role": "user", "content": content}]
    if model_key == "gpt-oss-20b":
        try:
            return tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True, reasoning_effort="low"
            )
        except TypeError:
            pass
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [6]:
# ============================================================
# FACTSCORE: Install (excluding Python 3.13-incompatible pysqlite-binary)
# ============================================================
!pip uninstall -y factscore -q
!pip install -q -U pip setuptools wheel
!rm -rf /content/OpenFActScore
!git clone -q https://github.com/lflage/OpenFActScore.git /content/OpenFActScore
!grep -v -i "pysqlite" /content/OpenFActScore/requirements.txt > /content/OpenFActScore/requirements_filtered.txt
!pip install -q -r /content/OpenFActScore/requirements_filtered.txt
!pip install -q -e /content/OpenFActScore --no-deps --no-build-isolation

sys.path.insert(0, "/content/OpenFActScore")
import sqlite3
print("sqlite3 OK:", sqlite3.sqlite_version)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu130 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Checking if build backend supports build_editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for OpenFactscore (pyproject.toml) ... done
sqlite3 OK: 3.45.1


In [7]:
# ============================================================
# FACTSCORE: Patch Llama3LM.py to use uniform bnb NF4 4-bit
# (upstream loads the judge in full precision, ~32GB per call)
# ============================================================
llama3lm_path = "/content/OpenFActScore/factscore/Llama3LM.py"

with open(llama3lm_path, "r") as f:
    content = f.read()

old_import = "from transformers import AutoModelForCausalLM\nfrom transformers import AutoTokenizer"
new_import = "from transformers import AutoModelForCausalLM\nfrom transformers import AutoTokenizer\nfrom transformers import BitsAndBytesConfig"

old_load = '''    def load_model(self):
        if self.model_dir:
            self.model = AutoModelForCausalLM.from_pretrained(self.model_dir).to("cuda")
        else:
            self.model = AutoModelForCausalLM.from_pretrained(self.model_name).to("cuda")'''

new_load = '''    def load_model(self):
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        if self.model_dir:
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_dir, quantization_config=bnb_config, device_map={"": 0},
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name, quantization_config=bnb_config, device_map={"": 0},
            )'''

if old_load in content:
    content = content.replace(old_import, new_import)
    content = content.replace(old_load, new_load)
    with open(llama3lm_path, "w") as f:
        f.write(content)
    print("Patched Llama3LM.py.")
else:
    print("Pattern not found — checking if already patched...")
    assert "quantization_config=bnb_config" in content, "Patch missing AND pattern not found — inspect file manually"
    print("Already patched (fresh clone must have been pre-patched, or this is a repeat run). OK.")

Patched Llama3LM.py.


In [8]:
# ============================================================
# FACTSCORE: Knowledge source (via Hugging Face mirror)
# ============================================================
FACTSCORE_CACHE_DIR = "/content/drive/MyDrive/thesis_results/factscore_cache"
os.makedirs(FACTSCORE_CACHE_DIR, exist_ok=True)

from huggingface_hub import hf_hub_download

db_path = os.path.join(FACTSCORE_CACHE_DIR, "enwiki-20230401.db")
if not os.path.exists(db_path) or os.path.getsize(db_path) < 1_000_000:
    print("Downloading enwiki-20230401.db (~21.9GB)...")
    downloaded = hf_hub_download(
        repo_id="lflage/wiki-talks-1", filename="enwiki-20230401.db",
        repo_type="dataset", local_dir=FACTSCORE_CACHE_DIR,
    )
    if downloaded != db_path:
        shutil.move(downloaded, db_path)

size_gb = os.path.getsize(db_path) / 1e9
print(f"Knowledge source: {size_gb:.2f} GB")
assert size_gb > 15, "Download incomplete"

Knowledge source: 21.92 GB


In [9]:
# ============================================================
# FACTSCORE: Demos (durable, Drive-backed)
# ============================================================
demos_dir = os.path.join(FACTSCORE_CACHE_DIR, "demos")

if not os.path.exists(os.path.join(demos_dir, "demons.json")):
    os.makedirs(demos_dir, exist_ok=True)
    demos_zip_id = "1IseEAflk1qqV0z64eM60Fs3dTgnbgiyt"
    zip_path = os.path.join(FACTSCORE_CACHE_DIR, "demos.zip")
    cmd = (
        f'wget --load-cookies /tmp/cookies.txt '
        f'"https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt '
        f'--keep-session-cookies --no-check-certificate '
        f"'https://docs.google.com/uc?export=download&id={demos_zip_id}' -O- | "
        f"sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\\1\\n/p')"
        f'&id={demos_zip_id}" -O {zip_path} && rm -rf /tmp/cookies.txt'
    )
    import subprocess
    subprocess.run([cmd], shell=True)
    subprocess.run([f"unzip -o {zip_path} -d {FACTSCORE_CACHE_DIR} && rm {zip_path}"], shell=True)

print("demos/ contents:")
!ls -la {demos_dir}

demos/ contents:
total 36
drwx------ 2 root root 4096 Oct  1 21:18 .
drwx------ 4 root root 4096 Oct  2 11:58 ..
-rw------- 1 root root 3151 May 23  2023 demons_complex.json
-rw------- 1 root root 8626 May 23  2023 demons_full.txt
-rw------- 1 root root 8099 May 23  2023 demons.json
-rw------- 1 root root 7903 May 23  2023 demons.txt


In [10]:
# ============================================================
# FACTSCORE: roberta_stopwords.txt (durable)
# ============================================================
stopwords_content = requests.get(
    "https://raw.githubusercontent.com/shmsw25/FActScore/main/roberta_stopwords.txt"
).text
for path in ["/content/roberta_stopwords.txt", "roberta_stopwords.txt"]:
    with open(path, "w") as f:
        f.write(stopwords_content)
print("roberta_stopwords.txt ready:", os.path.exists("roberta_stopwords.txt"))

roberta_stopwords.txt ready: True


In [11]:
# ============================================================
# FACTSCORE: FactScorer builder
# ============================================================
# IMPORTANT: Do not instantiate/score here. The OpenFActScore fork imports
# Llama3LM into atomic_facts.py, so the patched modules must be reloaded first.
from factscore.factscorer import FactScorer

FACTSCORE_GAMMA = 10

def build_factscorer():
    return FactScorer(
        afg_model=MODEL_REGISTRY[JUDGE_MODEL_KEY],
        afv_model=MODEL_REGISTRY[JUDGE_MODEL_KEY],
        is_npm=False,
        data_dir=FACTSCORE_CACHE_DIR,
        cache_dir=FACTSCORE_CACHE_DIR,
    )

print("FactScorer builder ready. Run the reload + smoke-test cells before scoring.")


FactScorer builder ready. Run the reload + smoke-test cells before scoring.


In [12]:
# ============================================================
# FACTSCORE: Load entity list (from manual upload or Drive cache)
# ============================================================
entities_cache_path = os.path.join(FACTSCORE_CACHE_DIR, "prompt_entities.txt")

if os.path.exists(entities_cache_path):
    with open(entities_cache_path) as f:
        entity_names = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(entity_names)} entities from cache.")
else:
    upload_path = "/content/prompt_entities.txt"
    assert os.path.exists(upload_path), "Upload prompt_entities.txt via the Colab sidebar first."
    with open(upload_path) as f:
        entity_names = [line.strip() for line in f if line.strip()]
    shutil.copy(upload_path, entities_cache_path)
    print(f"Loaded {len(entity_names)} entities, cached to Drive.")

assert len(entity_names) > 100, f"Only got {len(entity_names)} — check the file"

Loaded 500 entities from cache.


In [13]:
# ============================================================
# FACTSCORE: Pageview fetch (paced, retry-on-429, resumable)
# ============================================================
import time

def get_pageviews(entity, max_retries=3):
    title = entity.replace(" ", "_")
    url = (f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
           f"en.wikipedia/all-access/all-agents/{title}/daily/20230201/20230301")
    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers={"User-Agent": "thesis-research-bot/1.0 (contact: your_email@example.com)"})
            if r.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            items = r.json().get("items", [])
            return np.mean([it["views"] for it in items]) if items else 0
        except Exception:
            time.sleep(1)
    return 0

entity_freq = load_results("entity_freq") or {}
for i, name in enumerate(entity_names):
    if name not in entity_freq:
        entity_freq[name] = get_pageviews(name)
    time.sleep(0.3)
    if (i + 1) % 50 == 0:
        save_results("entity_freq", entity_freq)
        print(f"  {i+1}/{len(entity_names)} fetched")

save_results("entity_freq", entity_freq)
zero_count = sum(1 for v in entity_freq.values() if v == 0)
print(f"Done. Zero-view entities: {zero_count}/{len(entity_freq)}")

Saved entity_freq -> /content/drive/MyDrive/thesis_results/entity_freq.pkl (13:18:56)
  50/500 fetched
Saved entity_freq -> /content/drive/MyDrive/thesis_results/entity_freq.pkl (13:19:11)
  100/500 fetched
Saved entity_freq -> /content/drive/MyDrive/thesis_results/entity_freq.pkl (13:19:26)
  150/500 fetched
Saved entity_freq -> /content/drive/MyDrive/thesis_results/entity_freq.pkl (13:19:41)
  200/500 fetched
Saved entity_freq -> /content/drive/MyDrive/thesis_results/entity_freq.pkl (13:19:56)
  250/500 fetched
Saved entity_freq -> /content/drive/MyDrive/thesis_results/entity_freq.pkl (13:20:11)
  300/500 fetched
Saved entity_freq -> /content/drive/MyDrive/thesis_results/entity_freq.pkl (13:20:26)
  350/500 fetched
Saved entity_freq -> /content/drive/MyDrive/thesis_results/entity_freq.pkl (13:20:41)
  400/500 fetched
Saved entity_freq -> /content/drive/MyDrive/thesis_results/entity_freq.pkl (13:20:56)
  450/500 fetched
Saved entity_freq -> /content/drive/MyDrive/thesis_results/entity

In [14]:
# ============================================================
# FACTSCORE: Common 25-entity scoring set for ALL THREE models
# ============================================================
# The original experiment generated 50 biographies for Gemma/Phi-4 and
# 25 biographies for GPT-OSS. GPT-OSS's 25 were already selected as a
# subset of the original 50. We KEEP all existing biography generations
# and use only that existing 25-entity subset for model-level scoring.
#
# IMPORTANT: Do not regenerate biographies here.

TIER_NAMES = ["very_rare", "rare", "medium", "frequent", "very_frequent"]
valid_entity_freq = {name: freq for name, freq in entity_freq.items() if freq > 0}
freqs = np.array(list(valid_entity_freq.values()))
quantile_edges = np.quantile(freqs, [0.2, 0.4, 0.6, 0.8])

def assign_tier(freq):
    return TIER_NAMES[np.searchsorted(quantile_edges, freq)]

entity_tiers = {name: assign_tier(freq) for name, freq in valid_entity_freq.items()}

# Reconstruct the original 50-entity stratified sample deterministically.
random.seed(SEED)
FACTSCORE_N_PER_TIER = 10
sampled_entities = []
for tier in TIER_NAMES:
    pool = [n for n, t in entity_tiers.items() if t == tier]
    sampled_entities.extend(random.sample(pool, min(FACTSCORE_N_PER_TIER, len(pool))))

# The original GPT-OSS subset: first 5 entities from each tier of the
# original 50-entity sample = 25 entities total.
GPT_OSS_N_PER_TIER = 5
gpt_oss_sampled_entities = []
for tier in TIER_NAMES:
    tier_entities_in_main_sample = [e for e in sampled_entities if entity_tiers[e] == tier]
    gpt_oss_sampled_entities.extend(tier_entities_in_main_sample[:GPT_OSS_N_PER_TIER])

assert len(sampled_entities) == 50, f"Expected original sample of 50, got {len(sampled_entities)}"
assert len(gpt_oss_sampled_entities) == 25, f"Expected common sample of 25, got {len(gpt_oss_sampled_entities)}"

# THIS is now the only set used for model-level FActScore.
common_factscore_entities = list(gpt_oss_sampled_entities)

save_results("sampled_entities", sampled_entities)
save_results("gpt_oss_sampled_entities", gpt_oss_sampled_entities)
save_results("common_factscore_entities", common_factscore_entities)

print(f"Original generated sample: {len(sampled_entities)} entities")
print(f"Common scoring sample: {len(common_factscore_entities)} entities")
print("All three models will be scored on exactly the same 25 entities.")
print("No biography generation is performed by this cell.")


Saved sampled_entities -> /content/drive/MyDrive/thesis_results/sampled_entities.pkl (13:21:13)
Saved gpt_oss_sampled_entities -> /content/drive/MyDrive/thesis_results/gpt_oss_sampled_entities.pkl (13:21:13)
Saved common_factscore_entities -> /content/drive/MyDrive/thesis_results/common_factscore_entities.pkl (13:21:13)
Original generated sample: 50 entities
Common scoring sample: 25 entities
All three models will be scored on exactly the same 25 entities.
No biography generation is performed by this cell.


In [15]:
# ============================================================
# FACTSCORE: Biography generation — checkpointed per entity
# ============================================================
FACTSCORE_PROMPT_TEMPLATE = "Question: Tell me a bio of {entity}."
biographies = load_results("biographies") or {}

for model_key in SUBJECT_MODEL_KEYS:
    entities_for_model = gpt_oss_sampled_entities if model_key == "gpt-oss-20b" else sampled_entities
    biographies.setdefault(model_key, {})
    remaining = [e for e in entities_for_model if e not in biographies[model_key]]
    if not remaining:
        print(f"Skipping {model_key} — all {len(entities_for_model)} bios done")
        continue

    print(f"\n{'='*50}\nGenerating biographies with {model_key} ({len(remaining)} remaining)\n{'='*50}")
    model, tokenizer = load_model(model_key)

    for entity in remaining:
        prompt_text = FACTSCORE_PROMPT_TEMPLATE.format(entity=entity)
        chat_prompt = _build_chat_prompt(tokenizer, prompt_text, model_key=model_key)
        input_ids = tokenizer(chat_prompt, return_tensors="pt").input_ids.to("cuda")
        with torch.no_grad():
            out = model.generate(
                input_ids, max_new_tokens=400, do_sample=False, pad_token_id=tokenizer.pad_token_id,
            )
        bio_text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
        biographies[model_key][entity] = bio_text
        save_results("biographies", biographies)

    del model, tokenizer
    unload_model()
    gpu_status()

print("\nAll biographies generated.")
for model_key in SUBJECT_MODEL_KEYS:
    n_expected = len(gpt_oss_sampled_entities) if model_key == "gpt-oss-20b" else len(sampled_entities)
    print(f"{model_key}: {len(biographies.get(model_key, {}))}/{n_expected}")

Skipping gemma-3-12b — all 50 bios done
Skipping phi-4 — all 50 bios done

Generating biographies with gpt-oss-20b (25 remaining)
Loading gpt-oss-20b (openai/gpt-oss-20b)...


config.json:   0%|          | 0.00/1.81k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.20k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 27.9MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

... (5 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

Saved biographies -> /content/drive/MyDrive/thesis_results/biographies.pkl (04:14:56)
Saved biographies -> /content/drive/MyDrive/thesis_results/biographies.pkl (04:33:43)
Saved biographies -> /content/drive/MyDrive/thesis_results/biographies.pkl (04:52:29)
Saved biographies -> /content/drive/MyDrive/thesis_results/biographies.pkl (05:11:11)
Saved biographies -> /content/drive/MyDrive/thesis_results/biographies.pkl (05:29:58)
Saved biographies -> /content/drive/MyDrive/thesis_results/biographies.pkl (05:48:39)
Saved biographies -> /content/drive/MyDrive/thesis_results/biographies.pkl (06:07:25)
Saved biographies -> /content/drive/MyDrive/thesis_results/biographies.pkl (06:26:06)
Saved biographies -> /content/drive/MyDrive/thesis_results/biographies.pkl (06:44:51)
Saved biographies -> /content/drive/MyDrive/thesis_results/biographies.pkl (07:03:38)
Saved biographies -> /content/drive/MyDrive/thesis_results/biographies.pkl (07:22:20)
Saved biographies -> /content/drive/MyDrive/thesis_res

In [24]:
# ============================================================
# FACTSCORE: Use EXISTING biography generations — NO regeneration
# ============================================================
# The biographies were already generated. This cell deliberately does NOT
# call load_model(), model.generate(), or create new biographies.

biographies = load_results("biographies") or {}
assert common_factscore_entities, "Common 25-entity scoring set is empty."

for model_key in SUBJECT_MODEL_KEYS:
    missing = [
        e for e in common_factscore_entities
        if e not in biographies.get(model_key, {})
    ]
    assert not missing, (
        f"{model_key} is missing {len(missing)} biographies from the common 25-entity set: "
        f"{missing[:5]}"
    )
    print(f"{model_key}: {len(biographies[model_key])} existing biographies loaded; "
          f"{len(common_factscore_entities)}/25 required for scoring are present.")

print("Biography validation passed. No biographies were generated or regenerated.")


gemma-3-12b: 50 existing biographies loaded; 25/25 required for scoring are present.
phi-4: 50 existing biographies loaded; 25/25 required for scoring are present.
gpt-oss-20b: 25 existing biographies loaded; 25/25 required for scoring are present.
Biography validation passed. No biographies were generated or regenerated.


In [16]:
# ============================================================
# FACTSCORE: Second Llama3LM.py patch — fix the attribute-deletion
# bug (del self.model removed the attribute entirely, breaking the
# "if self.model is None" lazy-reload check on every call after the first)
# ============================================================

llama3lm_path = "/content/OpenFActScore/factscore/Llama3LM.py"

with open(llama3lm_path, "r") as f:
    content = f.read()

# 1. Ensure self.model always exists as an attribute, even before first load
old_init_tail = '''        self.logger = logging.getLogger(self.__class__.__name__)
        self.logits = False'''
new_init_tail = '''        self.logger = logging.getLogger(self.__class__.__name__)
        self.logits = False
        self.model = None
        self.tokenizer = None'''

# 2. Make unload_model safe to call even when no model is currently loaded
old_unload = '''    def unload_model(self):
        """
        Unloads the model and clears the memory.
        """
        del self.model  # Delete the model object
        torch.cuda.empty_cache()  # Free up GPU memory
        self.logger.debug("Model unloaded and GPU memory cleared.")'''

new_unload = '''    def unload_model(self):
        """
        Unloads the model and clears the memory.
        """
        if getattr(self, "model", None) is not None:
            del self.model
        self.model = None
        torch.cuda.empty_cache()  # Free up GPU memory
        self.logger.debug("Model unloaded and GPU memory cleared.")'''

assert old_init_tail in content, "init pattern not found — paste full __init__ so I can match exactly"
assert old_unload in content, "unload_model pattern not found — paste it exactly so I can match"

content = content.replace(old_init_tail, new_init_tail)
content = content.replace(old_unload, new_unload)

with open(llama3lm_path, "w") as f:
    f.write(content)
print("Patched Llama3LM.py: self.model now always exists; unload_model() is now idempotent.")

Patched Llama3LM.py: self.model now always exists; unload_model() is now idempotent.


In [17]:
# ============================================================
# FACTSCORE: Clear stale references + GPU memory before rebuilding fs
# ============================================================

for name in ["model", "tokenizer", "fs"]:
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
gpu_status()

fs = build_factscorer()

GPU: NVIDIA A100-SXM4-40GB
Allocated: 0.00 GB / Reserved: 0.00 GB


In [19]:
# ============================================================
# FACTSCORE: Reload patched modules in dependency order
# IMPORTANT: atomic_facts.py imports Llama3LM directly, so it must
# be reloaded after Llama3LM.py and before factscorer.py.
# ============================================================
import importlib
import factscore.Llama3LM
import factscore.atomic_facts
import factscore.factscorer

importlib.reload(factscore.Llama3LM)
importlib.reload(factscore.atomic_facts)
importlib.reload(factscore.factscorer)

from factscore.factscorer import FactScorer
from factscore.Llama3LM import Llama3LM

print("Reloaded: Llama3LM -> atomic_facts -> factscorer")


Reloaded: Llama3LM -> atomic_facts -> factscorer


In [27]:
# ============================================================
# FACTSCORE: Rebuild scorer and run a judge smoke test
# ============================================================
for name in ["model", "tokenizer", "fs"]:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    torch.cuda.synchronize()

# Build a completely new FactScorer after all dependent modules were reloaded.
fs = build_factscorer()

# Verify that both the scorer's verification LM and its atomic-fact LM
# are instances of the freshly patched Llama3LM class.
print("Verification LM:", type(fs.lm), "model attr:", hasattr(fs.lm, "model"))

# The atomic-fact generator is lazy, so instantiate it explicitly for the smoke test.
from factscore.atomic_facts import AtomicFactGenerator
fs.af_generator = AtomicFactGenerator(
    model_name=MODEL_REGISTRY[JUDGE_MODEL_KEY],
    demon_dir=os.path.join(FACTSCORE_CACHE_DIR, "demos"),
    key_path="",
    af_cache_file=os.path.join(FACTSCORE_CACHE_DIR, "InstructGPT.pkl"),
)
print("Atomic-fact LM:", type(fs.af_generator.lm), "model attr:", hasattr(fs.af_generator.lm, "model"))

assert isinstance(fs.lm, Llama3LM), "FactScorer is not using the patched Llama3LM class."
assert isinstance(fs.af_generator.lm, Llama3LM), "AtomicFactGenerator still holds a stale LM class."
assert hasattr(fs.lm, "model"), "Verification Llama3LM has no model attribute."
assert hasattr(fs.af_generator.lm, "model"), "Atomic-fact Llama3LM has no model attribute."

# Do not run a full FActScore here; this smoke test only verifies class wiring.
# The models are loaded lazily by generate().
fs.af_generator.lm.unload_model()
fs.lm.unload_model()
fs.af_generator = None   # was: del fs.af_generator — that removes the attribute,
                          # breaking get_score()'s "if self.af_generator is None" check
gc.collect()
torch.cuda.empty_cache()
print("FActScore module wiring smoke test PASSED.")


Verification LM: <class 'factscore.Llama3LM.Llama3LM'> model attr: True
Atomic-fact LM: <class 'factscore.Llama3LM.Llama3LM'> model attr: True
FActScore module wiring smoke test PASSED.


In [21]:
# ============================================================
# FACTSCORE: Final class-wiring diagnostic
# ============================================================
from factscore.Llama3LM import Llama3LM
from factscore.atomic_facts import AtomicFactGenerator
import inspect

print("Llama3LM __init__ contains model initialization:", "self.model = None" in inspect.getsource(Llama3LM.__init__))
print("Llama3LM unload_model is safe:", "getattr(self, \"model\", None)" in inspect.getsource(Llama3LM.unload_model))
print("AtomicFactGenerator Llama3LM class is current:", AtomicFactGenerator.__init__.__globals__["Llama3LM"] is Llama3LM)


Llama3LM __init__ contains model initialization: True
Llama3LM unload_model is safe: True
AtomicFactGenerator Llama3LM class is current: True


In [25]:
# ============================================================
# FACTSCORE: Initialise / restore scoring results
# ============================================================

# Reuse any previously saved FActScore results if they exist.
factscore_results = load_results("factscore_results") or {}

# Make sure all three subject models have an entry.
for model_key in SUBJECT_MODEL_KEYS:
    factscore_results.setdefault(model_key, {})

# Remove entries that do not contain a valid numeric score.
for model_key in SUBJECT_MODEL_KEYS:
    bad = [
        e for e, r in factscore_results[model_key].items()
        if not isinstance(r.get("score"), (int, float, np.number))
    ]

    for e in bad:
        del factscore_results[model_key][e]

    print(f"{model_key}: removed {len(bad)} incomplete entries")

save_results("factscore_results", factscore_results)

print("\nFActScore results initialised.")
for model_key in SUBJECT_MODEL_KEYS:
    print(f"{model_key}: {len(factscore_results[model_key])} existing scored entities")

gemma-3-12b: removed 0 incomplete entries
phi-4: removed 0 incomplete entries
gpt-oss-20b: removed 0 incomplete entries
Saved factscore_results -> /content/drive/MyDrive/thesis_results/factscore_results.pkl (13:25:37)

FActScore results initialised.
gemma-3-12b: 3 existing scored entities
phi-4: 0 existing scored entities
gpt-oss-20b: 0 existing scored entities


In [28]:
# ============================================================
# FACTSCORE: Score the SAME 25 biographies for ALL THREE models
# ============================================================
import traceback

# Do not score the extra 25 Gemma/Phi-4 biographies. They remain in the
# biography cache, but are excluded from this cross-model FActScore result.
for model_key in SUBJECT_MODEL_KEYS:
    factscore_results.setdefault(model_key, {})
    factscore_results[model_key] = {
        e: r for e, r in factscore_results[model_key].items()
        if e in common_factscore_entities
    }

for model_key in SUBJECT_MODEL_KEYS:
    entities_for_model = common_factscore_entities

    # Only treat an entry as completed if it contains a real numeric score.
    remaining = [
        e for e in entities_for_model
        if not isinstance(factscore_results[model_key].get(e, {}).get("score"), (int, float, np.number))
    ]

    if not remaining:
        print(f"Skipping {model_key} — all {len(entities_for_model)} common entities already scored")
        continue

    print(f"\n{'='*50}\nScoring {model_key} ({len(remaining)} remaining of common 25)\n{'='*50}")

    for i, entity in enumerate(remaining):
        bio = biographies[model_key][entity]
        try:
            result = fs.get_score(
                topics=[entity],
                generations=[bio],
                gamma=FACTSCORE_GAMMA,
            )

            score = result.get("score")
            if score is None or not np.isfinite(float(score)):
                raise RuntimeError(f"FActScore returned an invalid score: {result}")

            factscore_results[model_key][entity] = {
                "score": float(score),
                "init_score": result.get("init_score"),
                "respond_ratio": result.get("respond_ratio"),
                "num_facts": result.get("num_facts_per_response"),
                "decisions": result.get("decisions"),
            }
            print(f"  {entity}: score={float(score):.4f}")

        except Exception as e:
            factscore_results[model_key].pop(entity, None)
            print(f"  FAILED on {entity}: {type(e).__name__}: {e}")
            traceback.print_exc(limit=8)
            gc.collect()
            torch.cuda.empty_cache()

        save_results("factscore_results", factscore_results)
        if (i + 1) % 5 == 0 or (i + 1) == len(remaining):
            completed = sum(
                isinstance(r.get("score"), (int, float, np.number))
                for r in factscore_results[model_key].values()
            )
            print(f"  checkpoint: {i + 1}/{len(remaining)} — completed total: {completed}/{len(entities_for_model)}")

print("\nScoring pass complete.")
for model_key in SUBJECT_MODEL_KEYS:
    entities = factscore_results.get(model_key, {})
    scores = [float(r["score"]) for r in entities.values() if isinstance(r.get("score"), (int, float, np.number))]
    print(f"{model_key}: {len(scores)}/25 scored; mean FActScore={np.mean(scores):.3f}" if scores else f"{model_key}: 0/25 scored")



Scoring gemma-3-12b (22 remaining of common 25)
/content/drive/MyDrive/thesis_results/factscore_cache/retrieval-enwiki-20230401.json


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  Lemuel W. Joiner: score=0.0862
Saved factscore_results -> /content/drive/MyDrive/thesis_results/factscore_results.pkl (13:30:49)


... (213 outputs left out)

  Kajal Aggarwal: score=0.4889
Saved factscore_results -> /content/drive/MyDrive/thesis_results/factscore_results.pkl (14:44:46)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  Isla Fisher: score=0.4839
Saved factscore_results -> /content/drive/MyDrive/thesis_results/factscore_results.pkl (14:46:00)
  checkpoint: 25/25 — completed total: 22/25

Scoring pass complete.
gemma-3-12b: 24/25 scored; mean FActScore=0.355
phi-4: 23/25 scored; mean FActScore=0.447
gpt-oss-20b: 22/25 scored; mean FActScore=0.296


In [29]:
# ============================================================
# FACTSCORE: Retry pass for entities that failed during the main
# scoring loop (transient OOM from the judge's per-entity reload
# cycle) — exponential backoff, does not alter Cell 24
# ============================================================
import time

MAX_RETRIES_PER_ENTITY = 3

for model_key in SUBJECT_MODEL_KEYS:
    entities_for_model = common_factscore_entities
    remaining = [
        e for e in entities_for_model
        if not isinstance(factscore_results[model_key].get(e, {}).get("score"), (int, float, np.number))
    ]

    if not remaining:
        print(f"{model_key}: all 25 common entities already scored — nothing to retry")
        continue

    print(f"\n{'='*50}\nRetrying {model_key} ({len(remaining)} still missing)\n{'='*50}")

    for entity in remaining:
        bio = biographies[model_key][entity]
        success = False

        for attempt in range(1, MAX_RETRIES_PER_ENTITY + 1):
            try:
                result = fs.get_score(
                    topics=[entity],
                    generations=[bio],
                    gamma=FACTSCORE_GAMMA,
                )
                score = result.get("score")
                if score is None or not np.isfinite(float(score)):
                    raise RuntimeError(f"FActScore returned an invalid score: {result}")

                factscore_results[model_key][entity] = {
                    "score": float(score),
                    "init_score": result.get("init_score"),
                    "respond_ratio": result.get("respond_ratio"),
                    "num_facts": result.get("num_facts_per_response"),
                    "decisions": result.get("decisions"),
                }
                print(f"  {entity}: score={float(score):.4f} (attempt {attempt})")
                success = True
                break

            except Exception as e:
                print(f"  attempt {attempt}/{MAX_RETRIES_PER_ENTITY} FAILED on {entity}: {type(e).__name__}: {e}")
                gc.collect()
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
                time.sleep(5 * attempt)  # increasing backoff: 5s, 10s, 15s

        if not success:
            factscore_results[model_key].pop(entity, None)
            print(f"  {entity}: giving up after {MAX_RETRIES_PER_ENTITY} attempts — will need another pass")

        save_results("factscore_results", factscore_results)

print("\nRetry pass complete.")
for model_key in SUBJECT_MODEL_KEYS:
    entities = factscore_results.get(model_key, {})
    scores = [float(r["score"]) for r in entities.values() if isinstance(r.get("score"), (int, float, np.number))]
    print(f"{model_key}: {len(scores)}/25 scored; mean FActScore={np.mean(scores):.3f}" if scores else f"{model_key}: 0/25 scored")


Retrying gemma-3-12b (1 still missing)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  Frank Middlemass: score=0.2456 (attempt 1)
Saved factscore_results -> /content/drive/MyDrive/thesis_results/factscore_results.pkl (14:47:33)

Retrying phi-4 (2 still missing)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  Lemuel W. Joiner: score=0.0000 (attempt 1)
Saved factscore_results -> /content/drive/MyDrive/thesis_results/factscore_results.pkl (14:48:00)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

... (4 outputs left out)

  Deitrick Haddon: score=0.0408 (attempt 2)
Saved factscore_results -> /content/drive/MyDrive/thesis_results/factscore_results.pkl (14:49:32)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  Suthida: score=0.1739 (attempt 1)
Saved factscore_results -> /content/drive/MyDrive/thesis_results/factscore_results.pkl (14:50:44)

Retry pass complete.
gemma-3-12b: 25/25 scored; mean FActScore=0.350
phi-4: 25/25 scored; mean FActScore=0.438
gpt-oss-20b: 25/25 scored; mean FActScore=0.269


In [30]:
# ============================================================
# FACTSCORE: Export metadata and common-25 summary
# ============================================================
import csv

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S")
EXPORT_DIR = os.path.join(RESULTS_DIR, "exports", "factscore")
os.makedirs(EXPORT_DIR, exist_ok=True)

metadata = {
    "benchmark": "OpenFActScore",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "seed": SEED,
    "n_entities_original_generated_sample": len(sampled_entities),
    "n_entities_common_scoring_set": len(common_factscore_entities),
    "common_entities": common_factscore_entities,
    "tiers": TIER_NAMES,
    "entity_source": "unlabeled 500-entity set — labeled set's GitHub source was removed during data collection",
    "gamma": FACTSCORE_GAMMA,
    "afg_afv_judge": MODEL_REGISTRY[JUDGE_MODEL_KEY],
    "is_npm": False,
    "gpt_oss_quantization_note": "bf16+CPU-offload; MXFP4 unsupported on Ampere, 4-bit/8-bit conversion attempts corrupted weights",
    "knowledge_source": "enwiki-20230401.db via lflage/wiki-talks-1 (Hugging Face mirror)",
    "model_checkpoints": MODEL_REGISTRY,
    "biography_generation_note": "Existing biographies were reused; no biographies were regenerated for the common-25 evaluation.",
}
with open(os.path.join(EXPORT_DIR, f"metadata_{RUN_TIMESTAMP}.json"), "w") as f:
    json.dump(metadata, f, indent=2)

with open(os.path.join(EXPORT_DIR, f"summary_{RUN_TIMESTAMP}.csv"), "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["model", "entity", "tier", "factscore", "respond_ratio", "num_facts"])
    for model_key in SUBJECT_MODEL_KEYS:
        for entity in common_factscore_entities:
            r = factscore_results.get(model_key, {}).get(entity, {})
            writer.writerow([model_key, entity, entity_tiers.get(entity, "unknown"),
                             r.get("score"), r.get("respond_ratio"), r.get("num_facts")])

save_results(f"factscore_results_FINAL_{RUN_TIMESTAMP}", factscore_results)
print(f"Exported common-25 FActScore results to {EXPORT_DIR}")


Saved factscore_results_FINAL_2026-10-02_14-50-55 -> /content/drive/MyDrive/thesis_results/factscore_results_FINAL_2026-10-02_14-50-55.pkl (14:51:07)
Exported common-25 FActScore results to /content/drive/MyDrive/thesis_results/exports/factscore
